# 01 Baseline Model for Flow Separation Prediction

This notebook turns the starter flow separation CSV into a first usable machine learning pipeline.

The goals of this notebook are:
1. load the CSV
2. inspect the schema and confirm the target column
3. prepare the data for modeling
4. train baseline regression models
5. evaluate the models with MAE, RMSE, and R²
6. save the best model for later app integration

This notebook assumes the project repository has a structure similar to:

```text
aero-toolkit/
  data/
    processed/
      openFoamSimulationData3 - Sheet 1.csv
  models/
  notebooks/
```

The target variable is expected to be:

```text
separation_x_over_c
```

## 1. Import libraries and configure paths

This cell imports the core Python libraries needed for data loading, preprocessing, model training, evaluation, and model saving.

It also defines a few candidate CSV paths so the notebook can find the dataset even if the current working directory varies slightly.

In [ ]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (7, 5)

PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
]

CSV_CANDIDATES = []
for root in PROJECT_ROOT_CANDIDATES:
    CSV_CANDIDATES.extend([
        root / "data" / "processed" / "starter_flow_separation_data.csv",
        root / "data" / "processed" / "flow_separation_data.csv",
        root / "starter_flow_separation_data.csv",
    ])

MODEL_DIR = None
for root in PROJECT_ROOT_CANDIDATES:
    candidate = root / "models"
    if candidate.parent.exists():
        MODEL_DIR = candidate
        break

if MODEL_DIR is None:
    MODEL_DIR = Path.cwd() / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

CSV_CANDIDATES

## 2. Load the starter CSV

This cell searches for the starter CSV in a few likely locations, loads the first one it finds, and prints a quick preview.

If the notebook cannot find the file, update the `CSV_PATH` manually to match the actual location of the dataset.

In [ ]:
CSV_PATH = None
for candidate in CSV_CANDIDATES:
    if candidate.exists():
        CSV_PATH = candidate
        break

if CSV_PATH is None:
    raise FileNotFoundError(
        "Could not find the starter CSV. Please place it in data/processed/ "
        "or update CSV_PATH manually."
    )

df = pd.read_csv(CSV_PATH)
print(f"Loaded dataset from: {CSV_PATH}")
print(f"Shape: {df.shape}")
df.head()

## 3. Inspect the dataset structure

This cell checks the dataset shape, column names, data types, and missing values.

The purpose is to make sure the dataset matches the expected schema before any model training begins.

In [ ]:
print("Columns:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isnull().sum())

## 4. Confirm the target and feature columns

This cell locks the modeling schema for Version 1.

The target column is `separation_x_over_c`.

The feature columns are:
- tubercle_amplitude
- tubercle_wavelength
- tubercle_shape
- root_chord
- tip_chord
- sweep_angle
- angle_of_attack
- airspeed

This cell also checks that all required columns are present.

In [ ]:
TARGET_COLUMN = "separation_x_over_c"

FEATURE_COLUMNS = [
    "tubercle_amplitude",
    "tubercle_wavelength",
    "tubercle_shape",
    "root_chord",
    "tip_chord",
    "sweep_angle",
    "angle_of_attack",
    "airspeed",
]

required_columns = FEATURE_COLUMNS + [TARGET_COLUMN]
missing_required = [col for col in required_columns if col not in df.columns]

if missing_required:
    raise ValueError(
        f"Missing required columns: {missing_required}. "
        "Please fix the CSV schema before proceeding."
    )

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COLUMN].copy()

print("Target column confirmed:", TARGET_COLUMN)
print("Feature columns confirmed:")
for col in FEATURE_COLUMNS:
    print(" -", col)

## 5. Separate numeric and categorical features

This project contains both numeric and categorical inputs.

The categorical feature in Version 1 is `tubercle_shape`.

The numeric features will be imputed and scaled. The categorical feature will be imputed and one hot encoded.

In [ ]:
categorical_features = ["tubercle_shape"]
numeric_features = [col for col in FEATURE_COLUMNS if col not in categorical_features]

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 6. Create the preprocessing pipeline

This cell builds a preprocessing system using `ColumnTransformer`.

For numeric columns:
- impute missing values with the median
- scale the features

For categorical columns:
- impute missing values with the most frequent category
- one hot encode the categories

This lets the same pipeline be reused later inside the app.

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

preprocessor

## 7. Split the data into training and test sets

This cell creates an 80/20 train test split.

The training set is used to fit the model. The test set is kept separate so the evaluation reflects how the model performs on unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## 8. Build the first baseline model: Linear Regression

This is the simplest baseline and serves as a sanity check.

If this model performs very poorly, that is useful information. It suggests the problem may be strongly nonlinear or the dataset may need improvement.

In [ ]:
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

linear_model.fit(X_train, y_train)
linear_preds = linear_model.predict(X_test)

## 9. Evaluate the Linear Regression model

This cell computes:
- MAE
- RMSE
- R²

It also makes a predicted vs actual scatter plot. A stronger model should place points closer to the diagonal line.

In [ ]:
linear_mae = mean_absolute_error(y_test, linear_preds)
linear_rmse = np.sqrt(mean_squared_error(y_test, linear_preds))
linear_r2 = r2_score(y_test, linear_preds)

print("Linear Regression Metrics")
print(f"MAE:  {linear_mae:.4f}")
print(f"RMSE: {linear_rmse:.4f}")
print(f"R²:   {linear_r2:.4f}")

plt.figure()
plt.scatter(y_test, linear_preds)
min_val = min(y_test.min(), linear_preds.min())
max_val = max(y_test.max(), linear_preds.max())
plt.plot([min_val, max_val], [min_val, max_val])
plt.xlabel("Actual separation_x_over_c")
plt.ylabel("Predicted separation_x_over_c")
plt.title("Linear Regression: Actual vs Predicted")
plt.show()

## 10. Build a stronger baseline: Random Forest Regressor

This model can capture nonlinear relationships and interactions between features more easily than linear regression.

For structured tabular data like this, Random Forest is a strong early baseline.

In [ ]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=200,
            max_depth=None,
            random_state=42,
            n_jobs=-1,
        )),
    ]
)

rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

## 11. Evaluate the Random Forest model

This cell computes the same metrics used for Linear Regression so the two baselines can be compared directly.

In [ ]:
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))
rf_r2 = r2_score(y_test, rf_preds)

print("Random Forest Metrics")
print(f"MAE:  {rf_mae:.4f}")
print(f"RMSE: {rf_rmse:.4f}")
print(f"R²:   {rf_r2:.4f}")

plt.figure()
plt.scatter(y_test, rf_preds)
min_val = min(y_test.min(), rf_preds.min())
max_val = max(y_test.max(), rf_preds.max())
plt.plot([min_val, max_val], [min_val, max_val])
plt.xlabel("Actual separation_x_over_c")
plt.ylabel("Predicted separation_x_over_c")
plt.title("Random Forest: Actual vs Predicted")
plt.show()

## 12. Compare baseline model performance

This cell puts the metrics side by side in a table.

For MAE and RMSE, lower is better.
For R², higher is better.

In [ ]:
results_df = pd.DataFrame(
    [
        {
            "model": "Linear Regression",
            "MAE": linear_mae,
            "RMSE": linear_rmse,
            "R2": linear_r2,
        },
        {
            "model": "Random Forest Regressor",
            "MAE": rf_mae,
            "RMSE": rf_rmse,
            "R2": rf_r2,
        },
    ]
).sort_values(by="RMSE")

results_df

## 13. Save the stronger model for later app integration

This cell selects the better of the two baselines using RMSE and saves that full pipeline with `joblib`.

Saving the entire pipeline is helpful because it preserves both preprocessing and the fitted model together.

In [ ]:
best_model_name = results_df.iloc[0]["model"]

if best_model_name == "Linear Regression":
    best_model = linear_model
    best_model_filename = "baseline_linear_regression.joblib"
else:
    best_model = rf_model
    best_model_filename = "baseline_random_forest.joblib"

best_model_path = MODEL_DIR / best_model_filename
joblib.dump(best_model, best_model_path)

print("Best model selected:", best_model_name)
print("Saved to:", best_model_path)

## 14. Quick inference test on a single example

This cell creates one example input row and verifies that the saved pipeline can make a prediction.

This is useful because the app will eventually pass user inputs in a similar tabular format.

In [ ]:
example_input = pd.DataFrame(
    [
        {
            "tubercle_amplitude": 0.10,
            "tubercle_wavelength": 2.0,
            "tubercle_shape": "rectangular",
            "root_chord": 1.00,
            "tip_chord": 0.60,
            "sweep_angle": 10.0,
            "angle_of_attack": 5.0,
            "airspeed": 30.0,
        }
    ]
)

example_prediction = best_model.predict(example_input)[0]
print(f"Example predicted separation_x_over_c: {example_prediction:.4f}")

## 15. Recommended next steps

After this notebook runs successfully, the next project steps are:

1. inspect the dataset more deeply for outliers and unrealistic placeholder values
2. expand the dataset with more realistic cases
3. compare additional regression models if needed
4. load the saved model inside `src/inference.py`
5. replace the placeholder output in the Streamlit app with real model inference

This notebook is the bridge between the starter dataset and the app integration phase.